# Cuba water geographic names

Explore the preferred names, recorded alternatives and detailed feature types of
Cuba’s named water features. The registry includes streams, canals, ditches,
lakes, lagoons, reservoirs and other hydrological types; it is not limited to
rivers and lakes.

Names are independent of their physical geometry. A name object can be referenced
by several centerline reaches or polygon associations without becoming a new
name object for every segment.

Open this notebook from a local copy of the repository and run the cells in order
using **Pandas** and **PyArrow**. Set the working directory to `examples/`.
The examples below show a small preview and counts, rather than listing the
complete registry.


## Load the data

Read [Water Geographic Names](../data/water-geographic-names/) as a Parquet table
and display its size. This product has no geometry column or coordinate reference
system; spatial locations are supplied by the linked geometry datasets.


In [29]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("..")
WATER_NAMES = ROOT / "data/water-geographic-names/cuba_water_geographic_names.parquet"
names = pd.read_parquet(WATER_NAMES)

print(f"Rows: {len(names):,}")
print(f"Columns: {len(names.columns)}")


Rows: 3,687
Columns: 7


## Columns and data types

- `id` identifies a name object and provides the key used by geometry records.
- `preferred_name` is the display label. It uses a recorded complete form where available; otherwise a Spanish generic from the resolved feature type is added to the unchanged recorded name.
- `alternative_names` stores other recorded names of the same object as a JSON array. Original bare source forms are retained when a generic is added to the preferred label.
- `object_type` gives the detailed feature type, such as stream, canal, lagoon or reservoir.
- `system_ids` stores naming scopes as a JSON array. River identities are scoped to `MAIN_BAS`; a shared scope does not mean every tributary is the same named river.
- `source_observations` contains source and feature-ID references supporting the underlying recorded name. An added generic is derived from type, rather than attributed verbatim to the source.
- `alternative_name_object_references` links distinct objects that share applicability on specified geometry. These references are separate from aliases of the same object.

The next cell shows the loaded column types. JSON arrays are stored as strings;
use `json.loads(...)` to work with their contents. See [fields.json](../data/water-geographic-names/fields.json)
for the full field definitions.


In [30]:
structure = names.dtypes.astype(str).rename_axis("Column").reset_index(name="Data type")
print(structure.to_string(index=False))


                            Column Data type
                                id       str
                    preferred_name       str
                 alternative_names       str
                       object_type       str
                        system_ids       str
               source_observations       str
alternative_name_object_references       str


## Preview the records

Show the first five preferred names with their feature types and recorded
alternatives. Convert the JSON alternative-name arrays to readable text for this
preview; the loaded dataset remains available in `names`.


In [31]:
preview = names[["preferred_name", "object_type", "alternative_names"]].head().copy()
preview["alternative_names"] = preview["alternative_names"].map(
    lambda values: "; ".join(json.loads(values))
)
preview = preview.rename(columns={
    "preferred_name": "Geographic name",
    "object_type": "Feature type",
    "alternative_names": "Alternative names",
})
print(preview.to_string(index=False))


          Geographic name Feature type Alternative names
         Embalse Jucarral    reservoir                  
         Embalse La Ceiba    reservoir                  
Laguna de Mala Habitación       lagoon                  
           Embalse Caonao    reservoir      Presa Caonao
        Embalse Voladoras    reservoir                  


## Summarize feature types

Count name objects by their detailed feature type. These counts describe the
name registry, rather than numbers of physical reaches or polygon features.
Several reaches can reference one name object.


In [32]:
counts = (names["object_type"].fillna("Unresolved type").value_counts()
          .rename_axis("Feature type").reset_index(name="Name objects"))
print(counts.to_string(index=False))


         Feature type  Name objects
               stream          3097
            reservoir           200
                canal           149
               lagoon           101
                ditch            43
                 lake            30
              channel            23
           headwaters            18
                 pond            11
         stream mouth            10
          stream bend             2
    section of stream             2
abandoned watercourse             1


## Use names with geometry

Join `gazetteer_id` in [River Centerlines](../data/river-centerlines/) or
[Water-body Geometry](../data/water-body-geometry/) to `id` in this table.
`alternative_gazetteer_ids` contains additional assigned name references; decode
that JSON array to retrieve those names as well. An empty reference does not
remove the physical geometry from its dataset.

Use `alternative_names` to inspect recorded aliases belonging to one object.
Use `alternative_name_object_references` to inspect distinct objects sharing
geographic applicability. Shared applicability does not merge their identities.

Continue with [the hydrography map](hydrography_map.ipynb) to display the physical
layers, or [the ONEI river systems example](onei_river_systems.ipynb) to combine
names, the ONEI reference and complete connected networks.
